# Building the Frontier for AI with Databases

Oslo, 22 September 2026.

**This notebook is the fallback, not the demo.** The demo is the web app:

```bash
python -m app.app        # then http://127.0.0.1:5000
```

Every cell here runs the same function the matching panel calls, so the two
cannot drift apart. If a panel fails in the room, open this file, find the
matching heading, and run the cell. Outputs are committed, so even with no
network at all there is something true to show.

| Panel | Section here |
|---|---|
| Cosmos 1 to 5 | 2.1 to 2.7 |
| PostgreSQL 1 to 4 | 3.1 to 3.4 |
| Graph | 4 |
| Foundry, on the overview page | 5 |


## 1. Setup

The app package owns every query. Importing it here rather than restating the
SQL is the whole point: there is one implementation, and this notebook proves
it runs outside the web app too.

In [1]:
import sys, textwrap, time

sys.path.insert(0, "..")

from app import config, cosmos_store, foundry, graph_store, grounding, postgres_store, queries

print(f"Cosmos    {config.COSMOS_DATABASE}/{config.COSMOS_CONTAINER}  {config.COSMOS_DIMS} dims")
print(f"Postgres  {config.POSTGRES_DB}.{config.POSTGRES_TABLE}  {config.POSTGRES_DIMS} dims")
print(f"Foundry   {config.FOUNDRY_ACCOUNT}")
print(f"Models    {config.EMBEDDING_MODEL}, {config.EMBEDDING_SMALL_MODEL}")
print(f"Chat      {config.CHAT_MODEL} from the app, {config.PG_CHAT_MODEL} from inside PostgreSQL")


Cosmos    ops-db/tickets  3072 dims
Postgres  ops-db.tickets  1536 dims
Foundry   aif-aidb-swedencentral-001
Models    text-embedding-3-large, text-embedding-3-small
Chat      gpt-5.5 from the app, gpt-4.1-mini from inside PostgreSQL


In [2]:
def show(result, limit=5):
    """Print rows the way the UI does: the query, then the tickets."""
    rows, sql = result
    print(sql)
    print("-" * 100)
    if not rows:
        print("No rows. That is the result, not a failure.")
        return
    for row in rows[:limit]:
        score = row.get("score", row.get("distance", row.get("rrf", row.get("rank", ""))))
        if isinstance(score, float):
            score = f"{score:.4f}"
        head = f"{row['ticket_id']}  {row['domain']:<9} {row['error_code']}  {row['severity']:<8}"
        print(f"{str(score):<7} {head} {row['title']}")
        print(f"{'':<7} reported: {textwrap.shorten(row['description'], 92)}")
        if row.get("resolution"):
            print(f"{'':<7} fixed by: {textwrap.shorten(row['resolution'], 92)}")
        print()


def show_plain(result):
    rows, sql = result
    print(sql)
    print("-" * 100)
    for row in rows:
        print("  ", row)


## 2. Azure Cosmos DB for NoSQL

### 2.1 The data

Counts come from the database, not from the CSV.

In [3]:
stats = cosmos_store.stats()
print(f"{stats['total']} tickets")
for d in stats["by_domain"]:
    print(f"  {d['domain']:<10} {d['n']}")

sample = stats["sample"]
print()
print(f"{sample['ticket_id']}  {sample['domain']}/{sample['asset']}  {sample['error_code']}  {sample['severity']}")
print(f"  title:    {sample['title']}")
print(f"  reported: {sample['description']}")
print(f"  fixed by: {sample['resolution']}")

1000 tickets
  Energy     267
  Maritime   212
  Payments   239
  Retail     282

INC-2026-04000  Retail/DC Larvik  ERR-6410  High
  title:    Counts disagree between systems
  reported: The website is offering products the shelf does not actually have, and customers arrive to collect nothing.
  fixed by: Sync job failing silently on malformed records. Added validation and alerting, then reprocessed the backlog.


### 2.2 Text becomes numbers

One live embedding. The corpus was embedded ahead of time, but this is the proof
that the numbers are real and how long one costs.

In [4]:
text = "pump vibrating badly above 80 percent load"

started = time.time()
vector = foundry.embed_one(text, config.COSMOS_DIMS)
elapsed = int((time.time() - started) * 1000)

print(f"{config.EMBEDDING_MODEL}: {len(vector)} dimensions in {elapsed} ms")
print(f"\n\"{text}\"\n")
print([round(v, 6) for v in vector[:12]], f"... {len(vector) - 12} more")

text-embedding-3-large: 3072 dimensions in 1437 ms

"pump vibrating badly above 80 percent load"

[-0.015518, -0.000622, 0.006821, -0.001135, -0.013916, 0.004436, -0.026062, 0.0354, -0.023544, 0.003555, 0.029144, 0.009216] ... 3060 more


### 2.3 Keyword search

Literal matching against the full-text index, which is the same index hybrid
search fuses in 2.6. Perfect on an exact code, useless on a paraphrase.

`ERR-5012` is deliberately rare in this dataset, so it returns a handful of rows
rather than a quarter of the table.


In [5]:
print(f"ERR-5012 matches {cosmos_store.keyword_count('ERR-5012')} tickets in total\n")
show(cosmos_store.keyword_search("ERR-5012"), limit=3)

ERR-5012 matches 13 tickets in total

SELECT TOP @k c.ticket_id, c.opened_at, c.domain, c.asset, c.component, c.error_code, c.severity, c.status, c.team, c.title, c.description, c.resolution, c.resolved_hours
FROM c
WHERE FullTextContains(c.search_text, @term)
----------------------------------------------------------------------------------------------------
        INC-2026-04923  Maritime  ERR-5012  High     Main pump noisy at high output
        reported: Noticeable judder from the housing above three quarters output, easing off as soon as [...]
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

        INC-2026-04109  Maritime  ERR-5012  High     Main pump noisy at high output
        reported: The mounting feels alive under your hand when the machine is worked hard.
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

        INC-2026-04673  Maritime  ERR-5012  High  

In [6]:
# The same engine, asked for a symptom in words nobody happened to type.
print(f"'shaking' matches {cosmos_store.keyword_count('shaking')} tickets\n")
show(cosmos_store.keyword_search("shaking"))

'shaking' matches 0 tickets

SELECT TOP @k c.ticket_id, c.opened_at, c.domain, c.asset, c.component, c.error_code, c.severity, c.status, c.team, c.title, c.description, c.resolution, c.resolved_hours
FROM c
WHERE FullTextContains(c.search_text, @term)
----------------------------------------------------------------------------------------------------
No rows. That is the result, not a failure.


### 2.4 Vector search

The query that keyword search could not answer. Not one word of it appears in
the tickets it returns.

In [7]:
show(cosmos_store.vector_search("equipment shaking when running hard"), limit=3)

SELECT TOP @k c.ticket_id, c.opened_at, c.domain, c.asset, c.component, c.error_code, c.severity, c.status, c.team, c.title, c.description, c.resolution, c.resolved_hours,
       VectorDistance(c.embedding, @vec) AS score
FROM c
ORDER BY VectorDistance(c.embedding, @vec)
----------------------------------------------------------------------------------------------------
0.5634  INC-2026-04056  Maritime  ERR-5012  High     Vibration alarm on main pump during ramp-up
        reported: Something is clearly out of balance. The harder it runs the worse the racket gets.
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

0.5602  INC-2026-04109  Maritime  ERR-5012  High     Main pump noisy at high output
        reported: The mounting feels alive under your hand when the machine is worked hard.
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

0.5557  INC-2026-04500  Maritime  E

In [8]:
# The failure case, which is worth showing. A code carries no meaning to embed,
# so the nearest neighbours are unrelated tickets with confidently low distances.
show(cosmos_store.vector_search("ERR-5012"), limit=3)

SELECT TOP @k c.ticket_id, c.opened_at, c.domain, c.asset, c.component, c.error_code, c.severity, c.status, c.team, c.title, c.description, c.resolution, c.resolved_hours,
       VectorDistance(c.embedding, @vec) AS score
FROM c
ORDER BY VectorDistance(c.embedding, @vec)
----------------------------------------------------------------------------------------------------
0.3349  INC-2026-04698  Payments  ERR-4310  High     Settlement file rejected by clearing
        reported: Clearing would not take the submission, so nothing landed where it should have.
        fixed by: Disk exhausted on the batch host mid write. Cleared space, added an alert and reran cleanly.

0.3296  INC-2026-04190  Payments  ERR-4310  High     Settlement file rejected by clearing
        reported: Overnight run stopped halfway and left the day only partly posted.
        fixed by: Disk exhausted on the batch host mid write. Cleared space, added an alert and reran cleanly.

0.3296  INC-2026-04374  Payments  ERR-43

### 2.5 Filtered vector search

Meaning and metadata in one request. The filter is inside the query, so the top
results are the top results within scope, not a filtered copy of a wider list.

In [9]:
show(
    cosmos_store.filtered_vector_search(
        "equipment shaking when running hard", severity="Critical", domain="Maritime"
    ),
    limit=3,
)

SELECT TOP @k c.ticket_id, c.opened_at, c.domain, c.asset, c.component, c.error_code, c.severity, c.status, c.team, c.title, c.description, c.resolution, c.resolved_hours,
       VectorDistance(c.embedding, @vec) AS score
FROM c
WHERE c.severity = @severity AND c.domain = @domain
ORDER BY VectorDistance(c.embedding, @vec)
----------------------------------------------------------------------------------------------------
0.5557  INC-2026-04500  Maritime  ERR-5012  Critical Vibration alarm on main pump during ramp-up
        reported: Engineer reports the unit shudders badly once load goes past roughly 80 percent. [...]
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

0.4732  INC-2026-04916  Maritime  ERR-5012  Critical Main pump noisy at high output
        reported: Operator describes a rhythmic knocking that worsens the harder the unit works.
        fixed by: Bearing clearance out of tolerance. Replaced bearing set and re-a

### 2.6 Hybrid search

Both strategies in one clause, fused by the engine with RRF. Read the rank, not
the score: the score is an artefact of the fusion constant.

In [10]:
show(cosmos_store.hybrid_search("ERR-5012 vibration at high load"), limit=5)

SELECT TOP @k c.ticket_id, c.opened_at, c.domain, c.asset, c.component, c.error_code, c.severity, c.status, c.team, c.title, c.description, c.resolution, c.resolved_hours
FROM c
ORDER BY RANK RRF(
    FullTextScore(c.search_text, @term),
    VectorDistance(c.embedding, @vec)
)
----------------------------------------------------------------------------------------------------
1       INC-2026-04500  Maritime  ERR-5012  Critical Vibration alarm on main pump during ramp-up
        reported: Engineer reports the unit shudders badly once load goes past roughly 80 percent. [...]
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

2       INC-2026-04056  Maritime  ERR-5012  High     Vibration alarm on main pump during ramp-up
        reported: Something is clearly out of balance. The harder it runs the worse the racket gets.
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

3  

### 2.7 The same model, two different searches

Both sides are grounded, so the variable is the retrieval rather than the model.
The left side gets whatever the keyword index returned, which is nothing, and
reports truthfully that there is no record. The right side gets the same
question answered by meaning. Cited ticket IDs are checked against the database.


In [11]:
answer = grounding.compare(queries.GROUNDED_QUESTIONS[0]["value"])

for side, heading in (("keyword", "GROUNDED IN KEYWORD SEARCH"), ("semantic", "GROUNDED IN VECTOR SEARCH")):
    rows = answer[side]["rows"]
    print("=" * 100)
    print(f"{heading}  ({len(rows)} tickets handed to the model)")
    print("=" * 100)
    print(textwrap.fill(answer[side]["answer"], 98))
    cites = answer[side]["citations"]
    print("\ncited:", ", ".join(
        f"{c['ticket_id']} {'exists' if c['exists'] else 'DOES NOT EXIST'}" for c in cites
    ) or "nothing")
    print()


GROUNDED IN KEYWORD SEARCH  (0 tickets handed to the model)
We have no record of this happening before.

cited: nothing

GROUNDED IN VECTOR SEARCH  (8 tickets handed to the model)
Yes. Several ERR-5012 pump vibration cases above ~80%/high load were recorded.  Most were fixed by
correcting **soft foot on the mounting**: shim the base, re-torque base bolts, then retest at full
load with no recurrence (INC-2026-04500, INC-2026-04248, INC-2026-04056, INC-2026-04673,
INC-2026-04923, INC-2026-04109).  There were also similar high-load vibration cases caused by
**bearing clearance out of tolerance**; those were fixed by replacing the bearing set and re-
aligning the coupling (INC-2026-04372, INC-2026-04916).  Recommended first check: mounting soft
foot/base bolt condition, then bearing clearance/coupling alignment if vibration remains.

cited: INC-2026-04500 exists, INC-2026-04248 exists, INC-2026-04056 exists, INC-2026-04673 exists, INC-2026-04923 exists, INC-2026-04109 exists, INC-2026-0437

## 3. Azure Database for PostgreSQL

The same tickets, a different idea. In Cosmos the application does AI to the
data. Here the data does AI to itself.

### 3.1 Ask in plain English

Read the SQL. There is no vector in the request. PostgreSQL calls the embedding
model itself, using the server's managed identity, and this notebook holds no key.

In [13]:
show(postgres_store.plain_english_search("vibration and noise from a pump at high load"), limit=3)

WITH question AS (
    SELECT azure_openai.create_embeddings('text-embedding-3-large', 'vibration and noise from a pump at high load', dimensions => 1536)::vector AS v
)
SELECT ticket_id, opened_at, domain, asset, component, error_code, severity, status, team, title, description, resolution, resolved_hours,
       round((embedding <=> question.v)::numeric, 4) AS distance
FROM tickets, question
ORDER BY embedding <=> question.v
LIMIT 5
----------------------------------------------------------------------------------------------------
0.3312  INC-2026-04923  Maritime  ERR-5012  High     Main pump noisy at high output
        reported: Noticeable judder from the housing above three quarters output, easing off as soon as [...]
        fixed by: Soft foot on the mounting. Shimmed and re-torqued base bolts, retested at full load [...]

0.3323  INC-2026-04673  Maritime  ERR-5012  High     Vibration alarm on main pump during ramp-up
        reported: Noticeable judder from the housing above t

### 3.2 Join meaning to relational data

A similarity search is an ordinary result set, so ordinary SQL can aggregate
over it. This is the question a bolt-on vector store cannot answer.

In [14]:
show_plain(postgres_store.semantic_aggregate("payments are failing at the checkout"))

WITH matches AS (
    SELECT ticket_id, domain, team, severity, resolved_hours
    FROM tickets
    ORDER BY embedding <=> azure_openai.create_embeddings('text-embedding-3-large', 'payments are failing at the checkout', dimensions => 1536)::vector
    LIMIT 50
)
SELECT domain,
       team,
       count(*)                                                AS tickets,
       round(avg(resolved_hours)::numeric, 1)                  AS avg_hours,
       count(*) FILTER (WHERE severity IN ('Critical','High')) AS urgent
FROM matches
GROUP BY domain, team
ORDER BY tickets DESC, avg_hours DESC
----------------------------------------------------------------------------------------------------
   {'domain': 'Payments', 'team': 'Payments Platform', 'tickets': 19, 'avg_hours': 3.3, 'urgent': 18}
   {'domain': 'Payments', 'team': 'Core Banking', 'tickets': 18, 'avg_hours': 4.1, 'urgent': 17}
   {'domain': 'Payments', 'team': 'Fraud Engineering', 'tickets': 12, 'avg_hours': 3.0, 'urgent': 9}
   {'domai

### 3.3 Hybrid, by hand

The fusion Cosmos did in one clause, written out. It works, and you own every
line of it. That is the trade, in both directions.

In [15]:
show(postgres_store.hybrid_by_hand("pump vibration ERR-5012"), limit=3)

WITH semantic AS (
    SELECT ticket_id,
           row_number() OVER (ORDER BY embedding <=> azure_openai.create_embeddings('text-embedding-3-large', 'pump vibration ERR-5012', dimensions => 1536)::vector) AS rank
    FROM tickets
    LIMIT 50
),
lexical AS (
    SELECT ticket_id,
           row_number() OVER (
               ORDER BY ts_rank_cd(search_tsv, websearch_to_tsquery('english', 'pump vibration ERR-5012')) DESC
           ) AS rank
    FROM tickets
    WHERE search_tsv @@ websearch_to_tsquery('english', 'pump vibration ERR-5012')
    LIMIT 50
)
SELECT t.ticket_id, t.opened_at, t.domain, t.asset, t.component, t.error_code, t.severity, t.status, t.team, t.title, t.description, t.resolution, t.resolved_hours,
       round((coalesce(1.0/(60+s.rank), 0) + coalesce(1.0/(60+l.rank), 0))::numeric, 5) AS rrf
FROM tickets t
LEFT JOIN semantic s ON s.ticket_id = t.ticket_id
LEFT JOIN lexical  l ON l.ticket_id = t.ticket_id
WHERE s.ticket_id IS NOT NULL OR l.ticket_id IS NOT NULL
ORDER 

### 3.4 Retrieval and reasoning in one statement

Cosmos did this too, but the orchestration lived in Python: embed, search, build
a prompt, call the model. Here all four steps are the query. The embedding call
sits in its own CTE, otherwise the planner evaluates it once per row.

`azure_ai.generate` sends `temperature = 0.2` and offers no way to override it,
so this uses a deployment that still accepts a temperature.


In [16]:
rows, sql = postgres_store.answer_in_sql("payments are failing at the checkout")

print(sql)
print("-" * 100)
print(textwrap.fill(rows[0]["answer"], 98))

cited = postgres_store.tickets_exist(sorted({t for t in grounding.TICKET_ID.findall(rows[0]["answer"])}))
print("\ncited:", ", ".join(
    f"{c['ticket_id']} {'exists' if c['exists'] else 'DOES NOT EXIST'}" for c in cited
) or "nothing")


WITH question AS (
    SELECT 'payments are failing at the checkout' AS q
), asked AS (
    SELECT q, azure_openai.create_embeddings(
                  'text-embedding-3-large', q,
                  dimensions => 1536)::vector AS v
    FROM question
), matches AS (
    SELECT ticket_id, title, description, resolution
    FROM tickets, asked
    WHERE resolution IS NOT NULL AND resolution <> ''
    ORDER BY embedding <=> asked.v
    LIMIT 8
), context AS (
    SELECT string_agg(ticket_id || ' | ' || title
                      || E'\n  reported: ' || description
                      || E'\n  fixed by:  ' || resolution, E'\n') AS tickets
    FROM matches
)
SELECT azure_ai.generate(
           'Tickets from our system:' || E'\n' || context.tickets
           || E'\n\nQuestion: ' || asked.q,
           model        => 'gpt-4.1-mini',
           system_prompt => 'You are an operations support assistant. Answer in at '
                            'most 100 words, using only the tickets prov

## 4. The graph, in the same server

Apache AGE, in the same PostgreSQL instance that just did the vector search.

Two walks run here. One follows the fault code, which was already a column, so
SQL could have answered it alone. The other follows `SIMILAR_TO`, an edge built
from the embeddings rather than from a foreign key, and that is the one that
reaches assets sharing no code, no component and no domain with the question.


In [17]:
result = graph_store.expand("the terminal freezes in the middle of a transaction")

print(result["cypher"])
print("-" * 100)
print(f"seeded from {len(result['seeds'])} tickets found by vector search, all carrying {result['code']}\n")
print(f"{'asset':<22} {'domain':<10} {'this code':>9} {'similar':>8} {'avg hours':>10}  reached by")
for a in result["assets"]:
    how = "meaning only" if a["by_meaning_only"] else ("search" if a["in_search"] else "fault code")
    hours = "still open" if a["avg_hours"] is None else f"{a['avg_hours']}"
    print(f"{a['asset']:<22} {a['domain']:<10} {a['tickets'] or '':>9} {a['related'] or '':>8} {hours:>10}  {how}")

meaning = [a["asset"] for a in result["meaning_only"]]
print(f"\nreached only over the SIMILAR_TO edge: {', '.join(meaning) if meaning else 'none this time'}")


MATCH (c:Code {code: 'ERR-6205'})<-[:CODED]-(t:Ticket)-[:ON]->(a:Asset)
RETURN a.name, a.domain, count(t), avg(t.hours)

MATCH (c:Code {code: 'ERR-6205'})<-[:CODED]-(:Ticket)
      -[:SIMILAR_TO]->(t:Ticket)-[:ON]->(a:Asset)
RETURN a.name, a.domain, count(t), avg(t.hours)
----------------------------------------------------------------------------------------------------
seeded from 3 tickets found by vector search, all carrying ERR-6205

asset                  domain     this code  similar  avg hours  reached by
Store 031 Bergen       Retail            25                 5.2  search
Store 014 Oslo         Retail            23        3        4.7  fault code
Store 022 Trondheim    Retail            23                 5.4  search
DC Larvik              Retail            14       11        5.7  fault code
DC Vestby              Retail            13                 5.0  fault code
MV Bergen Star         Maritime                   11        3.8  meaning only
MV Havbris             Maritime

## 5. One Foundry account, three clients

My app for embeddings, my app for chat, and PostgreSQL itself through its own
managed identity. Same account, no keys anywhere.


In [18]:
import json, subprocess

raw = subprocess.run(
    ["az", "cognitiveservices", "account", "deployment", "list",
     "-g", config.RESOURCE_GROUP, "-n", config.FOUNDRY_ACCOUNT, "-o", "json"],
    capture_output=True, text=True, check=True,
).stdout

for d in sorted(json.loads(raw), key=lambda d: d["name"]):
    model = d["properties"]["model"]
    sku = d.get("sku", {})
    print(f"{d['name']:<24} {model['name']:<24} v{model['version']:<12} {sku.get('name')} capacity {sku.get('capacity')}")

gpt-4.1-mini             gpt-4.1-mini             v2025-04-14   GlobalStandard capacity 50
gpt-5.5                  gpt-5.5                  v2026-04-24   GlobalStandard capacity 100
text-embedding-3-large   text-embedding-3-large   v1            GlobalStandard capacity 50
text-embedding-3-small   text-embedding-3-small   v1            GlobalStandard capacity 120


In [19]:
# The third client, proved from inside the database.
conn = postgres_store._session()
with conn.cursor() as cur:
    cur.execute("SELECT azure_ai.get_setting('azure_openai.auth_type')")
    auth = cur.fetchone()[0]
    cur.execute("SELECT azure_ai.get_setting('azure_openai.endpoint')")
    endpoint = cur.fetchone()[0]
    cur.execute("SELECT array_length(azure_openai.create_embeddings(%s, 'live from inside postgres'), 1)",
                (config.EMBEDDING_SMALL_MODEL,))
    dims = cur.fetchone()[0]

print(f"PostgreSQL authenticates to Foundry with: {auth}")
print(f"endpoint: {endpoint}")
print(f"embedding returned from inside the database: {dims} dimensions")

PostgreSQL authenticates to Foundry with: managed-identity
endpoint: https://aif-aidb-swedencentral-001.openai.azure.com
embedding returned from inside the database: 1536 dimensions
